# Experiment 5 — Plausibility & Effect Interpretation

**PDF reference: Phase 5, Experiment 5**

Final sanity checks and practical interpretation of the estimated speed delta:

1. **Forest plot** — per-segment deltas with 95% CI and pooled estimate
2. **Effect size sanity** — typical aero gains are 0.5–2% at 40 kph; anything > 5% is suspicious
3. **I² heterogeneity** — are the per-segment deltas consistent, or does the effect vary?
4. **Paired vs unpaired segments** — does restricting to paired segments change the estimate?
5. **Interpretation table** — delta converted to sec/km and speed % at multiple reference powers

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.database import init_db, load_efforts, load_segments, load_bikes
from src.bike_delta import (
    prepare_delta_dataset, get_paired_segments,
    fit_baseline_model, compute_residuals,
    per_segment_delta, weighted_delta_summary,
    compute_i2, delta_to_sec_per_km,
)
from src.analytics import filter_outliers_by_power_speed

sns.set_theme(style='whitegrid', palette='Set2')
print('Imports OK')

In [ ]:
# ── Load and prepare ──────────────────────────────────────────────────────────
init_db()
raw_efforts = load_efforts()
segments = load_segments()
bikes_df = load_bikes()
bikes_dict = dict(zip(bikes_df['gear_id'], bikes_df['name'])) if not bikes_df.empty else {}

power_efforts = raw_efforts[raw_efforts['average_watts'].notna()].copy()
df = prepare_delta_dataset(power_efforts, segments, bikes_dict)
df, _ = filter_outliers_by_power_speed(df, z_threshold=2.0)

all_bikes = df['bike_name'].dropna().unique().tolist()

# Configure
REF_BIKE = all_bikes[0]
SPLINE_DF = 5
MIN_EFFORTS = 3

paired = get_paired_segments(df, all_bikes, min_efforts=MIN_EFFORTS)

# Fit model
model, design_info = fit_baseline_model(df, REF_BIKE, spline_df=SPLINE_DF)
df_resid = compute_residuals(df, model, design_info)

# Per-segment deltas
deltas_df = per_segment_delta(df_resid, paired, REF_BIKE, all_bikes)
summary_df = weighted_delta_summary(deltas_df)
i2_vals = compute_i2(deltas_df)

seg_names = dict(zip(segments['segment_id'], segments['name'])) \
    if 'name' in segments.columns else {}

print(f'Reference bike: {REF_BIKE}')
print(f'Paired segments: {len(paired)}')
print(f'Per-segment deltas computed: {len(deltas_df)}')

## 5.1 — Forest plot
Each row is one paired segment. The filled diamond is the inverse-variance weighted pooled estimate.

In [ ]:
other_bikes = [b for b in all_bikes if b != REF_BIKE]
palette = sns.color_palette('Set2', n_colors=len(other_bikes))

for idx, other in enumerate(other_bikes):
    pair_data = deltas_df[deltas_df['other_bike'] == other].copy()
    if pair_data.empty:
        print(f'No data for {REF_BIKE} → {other}')
        continue

    pair_data['seg_label'] = pair_data['segment_id'].map(lambda s: seg_names.get(s, str(s)))
    pair_data = pair_data.sort_values('delta', ascending=False).reset_index(drop=True)

    pair_key = f'{REF_BIKE} → {other}'
    pooled_row = summary_df[summary_df['bike_pair'] == pair_key]
    pooled_d = float(pooled_row['delta'].iloc[0]) if not pooled_row.empty else np.nan
    pooled_lo = float(pooled_row['ci_low'].iloc[0]) if not pooled_row.empty else np.nan
    pooled_hi = float(pooled_row['ci_high'].iloc[0]) if not pooled_row.empty else np.nan

    n = len(pair_data)
    fig, ax = plt.subplots(figsize=(10, max(4, n * 0.45 + 2)))

    y_positions = np.arange(n, 0, -1)
    ci = 1.96 * pair_data['se']
    ax.errorbar(
        x=pair_data['delta'], y=y_positions,
        xerr=ci, fmt='o', color=palette[idx],
        markersize=7, capsize=4, alpha=0.8, label='Segment estimate'
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(pair_data['seg_label'], fontsize=9)

    ax.axvline(0, color='grey', linestyle='--', lw=1.5)

    if not np.isnan(pooled_d):
        ax.axvline(pooled_d, color=palette[idx], linestyle=':', lw=2, label=f'Pooled: {pooled_d:+.5f}')
        ax.axvspan(pooled_lo, pooled_hi, alpha=0.12, color=palette[idx], label='Pooled 95% CI')
        diamond_y = 0.2
        diamond = plt.Polygon(
            [[pooled_d, diamond_y + 0.35], [pooled_hi, diamond_y],
             [pooled_d, diamond_y - 0.35], [pooled_lo, diamond_y]],
            closed=True, color=palette[idx]
        )
        ax.add_patch(diamond)

    ax.set_xlabel('Delta (speed/W¹⁄³) — positive = faster than reference', fontsize=10)
    ax.set_title(f'Forest plot: {pair_key}', fontsize=12)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.show()

## 5.2 — Effect size sanity check
Typical aero gains for a better road bike or aero bike are **0.5–2% speed** at 40 kph.  
Anything above **5%** is physically implausible for a bike comparison and suggests data problems or confounding.

In [ ]:
REF_SPEED_KMH = 30.0   # typical segment speed — adjust if your rides are faster/slower
REF_SPEED_MS  = REF_SPEED_KMH / 3.6

print(f'Reference speed for % calculation: {REF_SPEED_KMH:.0f} km/h')
print()

for ref_power in [150, 200, 250, 300]:
    for _, row in summary_df.iterrows():
        delta = float(row['delta'])
        speed_gain_kmh = delta * (ref_power ** (1.0 / 3.0))
        speed_gain_pct = speed_gain_kmh / REF_SPEED_KMH * 100
        flag = '⚠️  SUSPICIOUS' if abs(speed_gain_pct) > 5 else '✅'
        print(f'  {row["bike_pair"]:40s}  @{ref_power}W: '
              f'{speed_gain_kmh:+.2f} km/h  ({speed_gain_pct:+.1f}%)  {flag}')
    print()

## 5.3 — I² heterogeneity
I² measures how much of the variance between segment-level estimates is due to true heterogeneity vs sampling error.
- < 0.25 → Low: consistent effect across segments
- 0.25–0.75 → Moderate: some heterogeneity, possibly by terrain type
- > 0.75 → High: effect varies substantially — decompose by segment type

In [ ]:
print('=== I² heterogeneity ===')
for pair, i2 in i2_vals.items():
    label = '🟢 Low' if i2 < 0.25 else ('🟡 Moderate' if i2 < 0.75 else '🔴 High')
    print(f'  {pair:40s}  I² = {i2:.3f}  {label}')
    if i2 > 0.75:
        print('     → Decompose by segment grade / length / type before interpreting the pooled estimate.')

## 5.4 — Paired vs unpaired segment comparison
Does restricting to paired segments change the estimate substantially?
Large differences might indicate the paired segments are not representative.

In [ ]:
# All segments that any bike has attempted
all_seg_ids = df_resid['segment_id'].dropna().unique().tolist()

try:
    deltas_all = per_segment_delta(df_resid, all_seg_ids, REF_BIKE, all_bikes)
    summary_all = weighted_delta_summary(deltas_all)
    summary_paired = weighted_delta_summary(deltas_df)

    print('=== Paired vs unpaired comparison ===')
    for pair_key in summary_paired['bike_pair'].unique():
        p_row = summary_paired[summary_paired['bike_pair'] == pair_key]
        a_row = summary_all[summary_all['bike_pair'] == pair_key]
        if not p_row.empty and not a_row.empty:
            d_p = float(p_row['delta'].iloc[0])
            d_a = float(a_row['delta'].iloc[0])
            diff = abs(d_p - d_a)
            print(f'  {pair_key}')
            print(f'    Paired   : δ = {d_p:+.5f}  ({int(p_row["n_segments"].iloc[0])} segments)')
            print(f'    All segs : δ = {d_a:+.5f}  ({int(a_row["n_segments"].iloc[0])} segments)')
            flag = '⚠️  Large difference' if diff > 0.01 else '✅  Consistent'
            print(f'    Diff     : {diff:.5f}  {flag}')
            print()
except Exception as e:
    print(f'Paired vs all comparison failed: {e}')

## 5.5 — Interpretation table
Convert the pooled delta to human-readable speed gain at multiple reference power levels.

In [ ]:
REF_POWERS = [100, 150, 200, 250, 300, 350]
REF_SPEED_MS = 30.0 / 3.6

print('=== Interpretation table ===')
print(f'Reference speed: {REF_SPEED_MS * 3.6:.0f} km/h')
print()

for _, row in summary_df.iterrows():
    pair = str(row['bike_pair'])
    delta = float(row['delta'])
    n_seg = int(row['n_segments'])
    ci_lo = float(row['ci_low'])
    ci_hi = float(row['ci_high'])
    i2 = i2_vals.get(pair, 0.0)

    print(f'--- {pair}  (n={n_seg} segments, I²={i2:.2f}) ---')
    print(f'  Pooled δ = {delta:+.5f}  [95% CI: {ci_lo:+.5f}, {ci_hi:+.5f}]')
    print()
    print(f'  {"Power (W)":>12}  {"sec/km":>8}  {"km/h gain":>10}  {"% gain":>8}')
    print(f'  {"-"*12}  {"-"*8}  {"-"*10}  {"-"*8}')
    for pw in REF_POWERS:
        sec_km = delta_to_sec_per_km(delta, ref_power=float(pw), ref_speed_ms=REF_SPEED_MS)
        kmh_gain = delta * (pw ** (1.0 / 3.0))
        pct_gain = kmh_gain / (REF_SPEED_MS * 3.6) * 100
        print(f'  {pw:>12}  {sec_km:>+8.1f}  {kmh_gain:>+10.3f}  {pct_gain:>+8.2f}%')
    print()

print('Positive values = other bike is faster than reference.')
print('sec/km: seconds saved per km at given power. Negative = slower.')